# 🚗 CarValue AI — Car Price Prediction
**Dataset:** Quikr India Used Cars | **Model:** Linear Regression | **Author:** Your Name

---
This notebook walks through:
1. Data loading & initial exploration
2. Preprocessing & cleaning
3. Exploratory Data Analysis (EDA)
4. Feature engineering
5. Model training & evaluation

## 1 · Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib, json, os

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 110
print('✅ All libraries imported successfully')

## 2 · Load Raw Data

In [ ]:
df = pd.read_csv('../data/quikr_car.csv')
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Unique Values per Column ===')
print(df.nunique())

## 3 · Data Preprocessing & Cleaning

In [ ]:
# ── 3.1 Clean Price ──────────────────────────────────────────
print('Unique Price samples:', df['Price'].unique()[:10])

df = df[df['Price'] != 'Ask For Price'].copy()
df['Price'] = df['Price'].astype(str).str.replace(',', '', regex=False)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df.dropna(subset=['Price'], inplace=True)
df['Price'] = df['Price'].astype(int)
print(f'After price cleaning: {len(df)} rows')

In [ ]:
# ── 3.2 Clean kms_driven ────────────────────────────────────
print('Sample kms_driven values:', df['kms_driven'].unique()[:10])

df['kms_driven'] = df['kms_driven'].astype(str).str.replace(',', '', regex=False)
df['kms_driven'] = df['kms_driven'].str.extract(r'(\d+)')
df['kms_driven'] = pd.to_numeric(df['kms_driven'], errors='coerce')
df.dropna(subset=['kms_driven'], inplace=True)
df['kms_driven'] = df['kms_driven'].astype(int)
print(f'After kms_driven cleaning: {len(df)} rows')

In [ ]:
# ── 3.3 Clean year & fuel_type ───────────────────────────────
df.dropna(subset=['fuel_type'], inplace=True)
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df = df[(df['year'] >= 1990) & (df['year'] <= 2024)].copy()

# ── 3.4 Trim car name to 3 words ────────────────────────────
df['name'] = df['name'].astype(str).apply(lambda x: ' '.join(x.split()[:3]))

# ── 3.5 Remove outliers (IQR) ───────────────────────────────
Q1, Q3 = df['Price'].quantile(0.25), df['Price'].quantile(0.75)
IQR = Q3 - Q1
df = df[(df['Price'] >= Q1 - 1.5*IQR) & (df['Price'] <= Q3 + 1.5*IQR)].copy()

# ── 3.6 Drop duplicates ─────────────────────────────────────
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'✅ Final clean dataset: {df.shape}')
df.describe()

## 4 · Exploratory Data Analysis (EDA)

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Price'], bins=50, color='#00d4aa', edgecolor='none', alpha=0.85)
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price (₹)')

axes[1].hist(np.log1p(df['Price']), bins=50, color='#ff6b6b', edgecolor='none', alpha=0.85)
axes[1].set_title('Log(Price) Distribution')
axes[1].set_xlabel('log(Price + 1)')

plt.suptitle('Car Price — Raw vs Log Scale', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top companies
fig, ax = plt.subplots(figsize=(12, 6))
top = df['company'].value_counts().head(15)
ax.barh(top.index[::-1], top.values[::-1], color='#00d4aa', alpha=0.85)
ax.set_title('Top 15 Car Companies by Listings')
plt.tight_layout()
plt.show()

In [ ]:
# Fuel type distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['fuel_type'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%', 
                                     colors=['#00d4aa','#ff6b6b','#f9ca24','#6c5ce7'],
                                     startangle=140)
axes[0].set_title('Fuel Type Distribution')
axes[0].set_ylabel('')

order = df.groupby('fuel_type')['Price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='fuel_type', y='Price', order=order, palette='Set2', ax=axes[1])
axes[1].set_title('Price by Fuel Type')

plt.tight_layout()
plt.show()

In [ ]:
# Year trend & KMs scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

yearly = df.groupby('year')['Price'].median().reset_index()
axes[0].plot(yearly['year'], yearly['Price'], color='#00d4aa', lw=2.5, marker='o')
axes[0].fill_between(yearly['year'], yearly['Price'], alpha=0.15, color='#00d4aa')
axes[0].set_title('Median Price by Manufacturing Year')

sample = df.sample(min(500, len(df)), random_state=42)
sc = axes[1].scatter(sample['kms_driven'], sample['Price'], c=sample['year'],
                      cmap='plasma', alpha=0.7, s=25)
plt.colorbar(sc, ax=axes[1], label='Year')
axes[1].set_title('KMs Driven vs Price')
axes[1].set_xlabel('KMs Driven')
axes[1].set_ylabel('Price (₹)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr = df[['year', 'Price', 'kms_driven']].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

print('Key Observations:')
print(f'  year  ↔ Price : {corr.loc["year","Price"]:.3f}')
print(f'  kms   ↔ Price : {corr.loc["kms_driven","Price"]:.3f}')

## 5 · Model Training

In [ ]:
# Features & target
CAT_FEATURES = ['name', 'company', 'fuel_type']
NUM_FEATURES = ['year', 'kms_driven']
TARGET = 'Price'

X = df[CAT_FEATURES + NUM_FEATURES]
y = df[TARGET]

print(f'Feature matrix : {X.shape}')
print(f'Target vector  : {y.shape}')
X.head()

In [ ]:
# Train / test split — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'Train : {X_train.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')

In [ ]:
# Build sklearn Pipeline
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES),
    ('num', StandardScaler(), NUM_FEATURES),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model',        LinearRegression()),
])

# Train
pipeline.fit(X_train, y_train)
print('✅ Model trained!')

In [ ]:
# Evaluate
y_pred = np.maximum(pipeline.predict(X_test), 0)

mae  = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-6))) * 100

print(f'  MAE  : ₹ {mae:,.0f}')
print(f'  RMSE : ₹ {rmse:,.0f}')
print(f'  R²   :   {r2:.4f}')
print(f'  MAPE :   {mape:.2f} %')

In [ ]:
# Actual vs Predicted plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred, alpha=0.6, color='#00d4aa', s=30, edgecolors='none')
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Price (₹)')
ax.set_ylabel('Predicted Price (₹)')
ax.set_title(f'Actual vs Predicted  |  R² = {r2:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution
residuals = y_test.values - y_pred
plt.figure(figsize=(10, 5))
plt.hist(residuals, bins=40, color='#ff6b6b', edgecolor='none', alpha=0.8)
plt.axvline(0, color='white', lw=1.5, linestyle='--')
plt.xlabel('Residual (Actual − Predicted)')
plt.ylabel('Frequency')
plt.title('Residual Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Save model
os.makedirs('../model', exist_ok=True)
joblib.dump(pipeline, '../model/car_price_model.pkl')
print('✅ Model saved to ../model/car_price_model.pkl')

## 6 · Quick Prediction Test
Let's verify the model works with a sample input.

In [ ]:
sample_input = pd.DataFrame([{
    'name':       'Maruti Suzuki Alto',
    'company':    'Maruti',
    'fuel_type':  'Petrol',
    'year':       2016,
    'kms_driven': 35000,
}])

predicted = max(float(pipeline.predict(sample_input)[0]), 10_000)
print(f'Predicted price: ₹ {predicted:,.0f}')